# Mini Survey Data Cleaning

This notebook cleans the Group 3 mini survey CSV so it is ready for final processing and analysis. The cleaning steps preserve the original raw file, standardize column names, parse timestamps, trim text values, remove duplicate rows, and extract the Drake album year into a separate numeric column.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent

RAW_PATH = PROJECT_DIR / 'data' / 'raw' / 'Mini Survey Group 3 Artists and Songs (Responses) - Form Responses 1.csv'
INTERIM_PATH = PROJECT_DIR / 'data' / 'interim' / 'Mini Survey Group 3 Artists and Songs (Responses) - Form Responses 1.csv'

RAW_PATH, INTERIM_PATH

(PosixPath('/Users/samarth.gaggar/COSMOS/26-the-data-miners-analysis/group_survey/data/raw/Mini Survey Group 3 Artists and Songs (Responses) - Form Responses 1.csv'),
 PosixPath('/Users/samarth.gaggar/COSMOS/26-the-data-miners-analysis/group_survey/data/interim/Mini Survey Group 3 Artists and Songs (Responses) - Form Responses 1.csv'))

## Load Raw Data

The raw CSV contains Google Form responses with long question text as column names.

In [3]:
raw = pd.read_csv(RAW_PATH)
raw.head()

,Timestamp,What is your gender?,What grade are you entering into?,What is your race?,Out of these options what is your favorite Artist?,What is the best Drake album?
0,7/7/2026 10:44:27,Female,Junior,White,Lady Gaga,"Honestly, Nevermind (2022)"
1,7/7/2026 10:50:37,Female,Senior,Asian,The Weeknd,Nothing Was the Same (2013)
2,7/7/2026 10:46:35,Female,Senior,South Asian,Olivia Rodrigo,Scorpion (2018)
3,7/7/2026 10:48:57,Female,Junior,Asian,Olivia Rodrigo,Scorpion (2018)
4,7/7/2026 10:45:15,Female,Senior,South Asian,Olivia Rodrigo,Views (2016)


In [4]:
raw.shape, raw.isna().sum(), raw.duplicated().sum()

((19, 6),
 Timestamp                                             0
 What is your gender?                                  0
 What grade are you entering into?                     0
 What is your race?                                    0
 Out of these options what is your favorite Artist?    0
 What is the best Drake album?                         0
 dtype: int64,
 np.int64(0))

## Clean Data

The cleaned dataset uses short snake_case column names, parsed datetime values, trimmed text, a stable response ID, and a separate album year column.

In [5]:
column_map = {
    'Timestamp': 'timestamp',
    'What is your gender?': 'gender',
    'What grade are you entering into?': 'grade',
    'What is your race?': 'race',
    'Out of these options what is your favorite Artist?': 'favorite_artist',
    'What is the best Drake album?': 'best_drake_album',
}

clean = raw.rename(columns=column_map).copy()

text_columns = clean.select_dtypes(include='object').columns
clean[text_columns] = clean[text_columns].apply(lambda col: col.str.strip())

clean['timestamp'] = pd.to_datetime(clean['timestamp'], format='%m/%d/%Y %H:%M:%S')
clean = clean.drop_duplicates().sort_values('timestamp').reset_index(drop=True)
clean.insert(0, 'response_id', range(1, len(clean) + 1))
clean['best_drake_album_year'] = clean['best_drake_album'].str.extract(r'\((\d{4})\)').astype('Int64')

clean.head()

/var/folders/3l/s_btp51j2pb52565cgwd20f00000gn/T/ipykernel_83911/1292086258.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = clean.select_dtypes(include='object').columns


,response_id,timestamp,gender,grade,race,favorite_artist,best_drake_album,best_drake_album_year
0,1,2026-07-07 10:42:58,Male,Senior,Asian,Travis Scott,Certified Lover Boy (2021),2021
1,2,2026-07-07 10:43:51,Male,Senior,South Asian,The Weeknd,If You're Reading This It's Too Late (2015),2015
2,3,2026-07-07 10:44:01,Male,Junior,Asian,Kendrick Lamar,Views (2016),2016
3,4,2026-07-07 10:44:02,Male,Junior,South Asian,Drake,For All the Dogs (2023),2023
4,5,2026-07-07 10:44:22,Male,Senior,Asian,The Weeknd,Take Care (2011),2011


## Validate Clean Data

These checks confirm the cleaned file has no missing required values, no duplicate records, and expected category values.

In [6]:
required_columns = [
    'response_id',
    'timestamp',
    'gender',
    'grade',
    'race',
    'favorite_artist',
    'best_drake_album',
    'best_drake_album_year',
]

assert clean.columns.tolist() == required_columns
assert clean[required_columns].isna().sum().sum() == 0
assert clean.duplicated().sum() == 0
assert clean['response_id'].is_unique
assert clean['gender'].isin(['Female', 'Male']).all()
assert clean['grade'].isin(['Junior', 'Senior']).all()
assert clean['best_drake_album_year'].between(2010, 2026).all()

clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   response_id            19 non-null     int64         
 1   timestamp              19 non-null     datetime64[us]
 2   gender                 19 non-null     str           
 3   grade                  19 non-null     str           
 4   race                   19 non-null     str           
 5   favorite_artist        19 non-null     str           
 6   best_drake_album       19 non-null     str           
 7   best_drake_album_year  19 non-null     Int64         
dtypes: Int64(1), datetime64[us](1), int64(1), str(5)
memory usage: 1.3 KB


## Save Clean Interim Data

The cleaned CSV is saved in `data/interim/`. It is now ready to be moved into `data/processed/` after final analysis decisions.

In [7]:
clean.to_csv(INTERIM_PATH, index=False)
INTERIM_PATH

PosixPath('/Users/samarth.gaggar/COSMOS/26-the-data-miners-analysis/group_survey/data/interim/Mini Survey Group 3 Artists and Songs (Responses) - Form Responses 1.csv')